<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Tokens and embeddings

**[Tokens and embeddings](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/)** · Deep Learning and Transformers Explained · Module 5, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn a ticket into tokens and token IDs, look up embedding rows with `nn.Embedding`, train the average-embedding network on Larkfield's tickets, and find the nearest words of a trained embedding, honestly.

| | |
|---|---|
| **Time** | About 30 minutes, including one minute of training |
| **Needs** | An internet connection for the first cells (about 2 MB of tickets). No account. Colab and Kaggle have PyTorch. |
| **You should know** | The vocabulary and the bag of words, from [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/). |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M05-L01-tokens-and-embeddings/tokens-and-embeddings-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Tokens and IDs

`tokenize` cuts a text into lower-case words, numbers and punctuation marks. The vocabulary gives each token that appears at least twice in the training tickets an ID. ID 0 is `<pad>` and ID 1 is `<unk>`, any unknown token.

> **Predict.** The ticket below ends with an order number. Which ID will it get?

Run the cell.

In [ ]:
vocab = Vocab([r["text"] for r in train_rows])

text = "Ugh, need to change trampoline delivery slot. Order LK-640933"
print(tokenize(text))
print(vocab.encode(text))
print(len(vocab), "entries;", vocab.itos[:2], "; ID of 'delivery':", vocab.stoi["delivery"])

> **Check.** You should see 13 tokens, the vocabulary size `2864`, and the order number `640933` as ID `1`: it appears in no training ticket.

## 2. One-hot vectors say nothing about similarity

A one-hot vector has one number per vocabulary entry, all 0 except a 1 at the token's ID.

> **Your turn.** Build the one-hot vectors of "refund" and "refunded" in `one_hot_refund` and `one_hot_refunded` (start from `torch.zeros(len(vocab))`). Then compute their dot product in `dot`.

In [ ]:
one_hot_refund = torch.zeros(len(vocab))
one_hot_refund[vocab.stoi["refund"]] = 1
one_hot_refunded = torch.zeros(len(vocab))
one_hot_refunded[vocab.stoi["refunded"]] = 1
dot = float(one_hot_refund @ one_hot_refunded)
print(one_hot_refund.shape, dot)

In [ ]:
expect('dot', dot, 0.0)

The dot product is 0 for **any** two different words: one-hot vectors make every pair of words equally different.

## 3. An embedding is a table lookup

`nn.Embedding(2864, 64)` is a table of 2,864 rows with 64 random numbers each; training will change them. `embed(ids)` returns the row of each ID. `padding_idx=0` keeps the `<pad>` row at zero.

Run the cell.

In [ ]:
from torch import nn

torch.manual_seed(0)
embed = nn.Embedding(len(vocab), 64, padding_idx=0)
ids = torch.tensor([vocab.encode(text)])        # one ticket
vectors = embed(ids)
show_shape("ids", ids)
show_shape("vectors", vectors)
print(torch.equal(vectors[0, 6], embed.weight[832]))  # token 7 is 'delivery', ID 832

> **Check.** `ids` has the shape (1, 13) and `vectors` (1, 13, 64): one row of 64 numbers per token. The last line prints `True`.

> **Your turn.** `token_ids` pads tickets with zeros to the same length. Make the token IDs of the first 8 training tickets, look them up, and store the shape of the result in `batch_shape` (as a tuple).

In [ ]:
batch = token_ids(train_rows[:8], vocab)
batch_shape = tuple(embed(batch).shape)
print(batch.shape, "->", batch_shape)

In [ ]:
expect('batch_shape', batch_shape, (8, 120, 64))

The first number is the batch of 8 tickets, the second the token positions of the longest of them (the course cuts tickets at 120 tokens), the last the 64 numbers of each embedding.

## 4. Train the average-embedding network

`AverageEmbeddingNet` looks up the rows of a ticket's tokens, averages them, and sends the average to one output layer with 5 team scores. The next cell trains it with the lesson's settings: 30 epochs, learning rate 0.01. It takes about one minute.

> **Predict.** Will it route more validation tickets right than the bag-of-words network (98.3% at its best epoch)?

In [ ]:
ids_train, ids_valid = token_ids(train_rows, vocab), token_ids(valid_rows, vocab)
set_seed(0)
model = AverageEmbeddingNet(len(vocab))
print(count_parameters(model), "parameters")
history = train(model, ids_train, labels(train_rows), ids_valid, labels(valid_rows), epochs=30, lr=0.01, log=None)
best = min(history, key=lambda h: h["valid_loss"])
print("best:", best)
print("last:", history[-1])

In [ ]:
expect('the best validation accuracy', best['valid_acc'], 0.98, tolerance=0.01)

> **Check.** The lesson's run: 183,621 parameters, best epoch 7 with 98.0% of the validation tickets right, then 96.8% after 30 epochs. Another computer can give slightly different numbers. It is **not** better than the bag of words: an average forgets the word order.

## 5. Nearest words

Scale every row to length 1. Then the dot product of two rows is their cosine similarity.

Run the cell.

In [ ]:
W = model.embed.weight.detach()
W = W / W.norm(dim=1, keepdim=True).clamp(min=1e-8)

def nearest(word, k=5):
    sims = W @ W[vocab.stoi[word]]
    top = sims.topk(k + 1).indices.tolist()
    return [(vocab.itos[i], round(sims[i].item(), 2)) for i in top if vocab.itos[i] != word][:k]

for word in ("return", "refund", "cracked", "password", "charged"):
    print(word, nearest(word))

Read the lists honestly: "charged" → "twice", "total", "charges" makes sense for the payment team, but "cracked" → "where", "along", "here" is noise. "cracked" appears in warranty and in delivery tickets almost equally, so its row points to no team.

> **Your turn.** Write `cosine(a, b)`, the cosine similarity of two words' rows of `W`, rounded to 2 decimals. Predict first: is "refund" close to "refunded"? Then compute `refund_refunded` and `cracked_broken`.

In [ ]:
def cosine(a, b):
    return round((W[vocab.stoi[a]] @ W[vocab.stoi[b]]).item(), 2)

refund_refunded = cosine("refund", "refunded")
cracked_broken = cosine("cracked", "broken")
print(refund_refunded, cracked_broken)

In [ ]:
expect('refund_refunded', refund_refunded, -0.28, tolerance=0.1)

In [ ]:
expect('cracked_broken', cracked_broken, 0.49, tolerance=0.1)

"refund" and "refunded" are **not** close: "refunded" appears only 28 times in the training tickets. Embeddings measure "used alike for this task", not "means the same". (The tolerance is wide, because another computer can train slightly different rows.)

## 6. Failure case: IDs as plain numbers

Tomás feeds the token IDs themselves to a network, as numbers between 0 and 1. Run the cell (about 20 seconds).

In [ ]:
def as_numbers(ids, width=120):
    x = torch.zeros(len(ids), width)
    x[:, :ids.shape[1]] = ids[:, :width].float()
    return x / len(vocab)

set_seed(0)
id_net = nn.Sequential(nn.Linear(120, 64), nn.ReLU(), nn.Linear(64, 5))
h = train(id_net, as_numbers(ids_train), labels(train_rows), as_numbers(ids_valid), labels(valid_rows), epochs=20, log=None)
print("validation accuracy:", h[-1]["valid_acc"])

> **Check.** About 0.29: no better than always answering "delivery" (0.275). An ID is a name, not an amount. Turn IDs into vectors first, with a one-hot vector or an embedding.

## Next

You can now turn text into token IDs, look up embeddings, train the average-embedding network and read its nearest words with care. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/) has the diagram, the embedding map and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Why word order and context matter](https://learning.nextia-ai.com/courses/deep-learning/m05/word-order-and-context/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.